In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import trange, tqdm
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoImageProcessor

/home/kpetrakis/projects/trunk-filtering/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_SIGLIP_ID = "google/siglip-so400m-patch14-384"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained(_SIGLIP_ID).to(device).eval()
processor = AutoImageProcessor.from_pretrained(_SIGLIP_ID, backend="torchvision")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 888/888 [00:00<00:00, 5718.79it/s]


In [4]:
'''
inference/trunk_export/
  crops/{key}.png        one per trunk detection, WITH context margin (lossless)
  meta.csv               key, stem, frame, x1..y2, crop offset, conf, depth_q25, box_h
  references.json        the annotator's fg/bg picks as keys (written at filter time)
inference/trunk_embeddings.npz   (already exists) current SigLIP embeddings
'''

DATA_ROOT = Path("../data/ab9cadb8-a4be-4357-b2d6-b781b625a2f9")
INFERENCE_ROOT = DATA_ROOT / "inference"
CROPS_ROOT = INFERENCE_ROOT / "trunk_export/crops"
meta_path = INFERENCE_ROOT / "trunk_export" / "meta.csv"

meta = pd.read_csv(meta_path)
npz = np.load(INFERENCE_ROOT / "trunk_embeddings.npz", allow_pickle=True)
siglip = dict(zip(npz["keys"].astype(str), npz["emb"]))                                  # key -> (1152,)
# print("Available arrays:", npz.files)
# for name in npz.files:
#   arr = npz[name]
#   print(f"\narray {name}:")
#   print("  Shape:", arr.shape)
#   print("  Dtype:", arr.dtype)
#   print("  Values:\n", arr[:5] if arr.ndim > 0 else arr)

refs = json.loads((INFERENCE_ROOT / "trunk_export/references.json").read_text())
meta["ref"] = meta["key"].map({**{k: "fg" for k in refs["fg"]}, **{k: "bg" for k in refs["bg"]}})  # fg / bg / NaN

In [5]:
# # for _, row in tqdm(meta.iterrows()):
# # for i in tqdm(range(50160)):
#   row = meta.iloc[i]
#   img = Image.open(CROPS_ROOT / f"{row.key}.png").convert("RGB") # this is the
#   bx1, by1 = row.x1 - row.crop_x1, row.y1 - row.crop_y1                     # box inside the crop
#   bx2, by2 = row.x2 - row.crop_x1, row.y2 - row.crop_y1
#   crop = img.crop((bx1, by1, bx2, by2))
# 
#   with torch.inference_mode():
#     inputs = processor(images=crop, return_tensors="pt").to(device)
#     emb = model.get_image_features(**inputs).pooler_output # (B, D)
#     emb = F.normalize(emb, p=2, dim=-1)

In [79]:
row = meta.iloc[114]
print(row)
img = Image.open(CROPS_ROOT / f"{row.key}.png").convert("RGB") # this is the 
img.mode
bx1, by1 = row.x1 - row.crop_x1, row.y1 - row.crop_y1                     # box inside the crop
bx2, by2 = row.x2 - row.crop_x1, row.y2 - row.crop_y1

crop = img.crop((bx1, by1, bx2, by2))
crop, siglip[row.key]

key          L_1782121838_p0000_chunk_005_000036_1116_864
stem                  L_1782121838_p0000_chunk_005_000036
frame                                                  36
x1                                                   1116
y1                                                    864
x2                                                   1457
y2                                                   2043
crop_x1                                               946
crop_y1                                               570
conf                                             0.775879
depth_q25                                        0.654821
box_h                                                1179
ref                                                    fg
Name: 114, dtype: object


(<PIL.Image.Image image mode=RGB size=341x1179>,
 array([-0.01749919,  0.0230463 , -0.01573422, ..., -0.00638481,
        -0.01270856, -0.01858864], shape=(1152,), dtype=float32))

In [80]:
# meta[meta['ref'] == "fg"]

In [ ]:
with torch.inference_mode():
  inputs = processor(images=crop, return_tensors="pt").to(device)
  emb = model.get_image_features(**inputs).pooler_output # (B, D)
  emb = F.normalize(emb, p=2, dim=-1)

# emb.flatten()
emb.flatten().numpy(), siglip[row.key]

tensor([-0.0178,  0.0249, -0.0161,  ..., -0.0071, -0.0144, -0.0162])

In [83]:
w = torch.from_numpy(siglip[row.key])
print(float(emb.flatten().cpu() @ w))          # cosine similarity (both are L2-normalized)

0.9986808896064758


In [7]:
fg_bank, bg_bank = [], []
for stem in refs['fg']:
  # print(f"{stem}: {siglip[stem]}")
  fg_bank.append(siglip[stem])

for stem in refs['bg']:
  bg_bank.append(siglip[stem])

fg_bank = torch.from_numpy(np.array(fg_bank))
bg_bank = torch.from_numpy(np.array(bg_bank))

In [33]:
SIGLIP_FLOOR  = 0.0    # tune: if BOTH prototype sims are below this → "looks like neither" → unknown
SIGLIP_MARGIN = 0.02 # |a - b| below this → too ambiguous → unknown
SIGLIP_TOPK = 3
def _topk_score(v, bank):
  """Mean of the top-k cosine similarities between v (D,) and the bank rows (K, D)."""
  import torch
  sims = bank @ v # (K,) cosine, all unit vectors
  return float(torch.topk(sims, k=min(SIGLIP_TOPK, len(sims))).values.mean())

# hits = {"fg": 0, "bg": 0, "unknown":0}
# for name, own, other in (("fg", fg_bank, bg_bank), ("bg", bg_bank, fg_bank)):
#   for i in range(len(own)):
#     rest = torch.cat([own[:i], own[i+1:]])
#     a, b = _topk_score(own[i], rest), _topk_score(own[i], other)
#     if a < SIGLIP_FLOOR and b < SIGLIP_FLOOR:
#         hits["unknown"] += 1
#     if abs(a - b) < SIGLIP_MARGIN:
#         hits["unknown"] += 1
#     hits[name] += int(a > b)

# hits

from collections import Counter

def decide(a, b):                                   # identical to _siglip_class2
    if a < SIGLIP_FLOOR and b < SIGLIP_FLOOR: return "unknown"
    if abs(a - b) < SIGLIP_MARGIN:            return "unknown"
    return "foreground" if a > b else "background"

conf = Counter()
for label, own, other in (("foreground", fg_bank, bg_bank), ("background", bg_bank, fg_bank)):
    for i in range(len(own)):
        rest = torch.cat([own[:i], own[i+1:]])
        fg_b, bg_b = (rest, other) if label == "foreground" else (other, rest)   # bank roles, not own/other
        a, b = _topk_score(own[i], fg_b), _topk_score(own[i], bg_b)
        conf[(label, decide(a, b))] += 1

pd.Series(conf).unstack(fill_value=0)               # rows = true label, cols = decision

,background,foreground,unknown
background,46,0,4
foreground,0,48,2


In [ ]:
# references se apostasi ligotero apo 30 frames
r = meta[meta.ref.notna()].sort_values("frame")
print(r.groupby("ref")["frame"].apply(lambda f: f.diff().lt(30).sum()))   # refs within 30 frames of another ref

ref
bg    24
fg    19
Name: frame, dtype: int64


In [23]:
def _siglip_class2(v, fg_bank, bg_bank):
  if v is None:
    return "unknown"
  a, b = _topk_score(v, fg_bank), _topk_score(v, bg_bank)
  if a < SIGLIP_FLOOR and b < SIGLIP_FLOOR:
    return "unknown"
  if abs(a - b) < SIGLIP_MARGIN:
    return "unknown"
  return "foreground" if a > b else "background"

dec = {str(k): _siglip_class2(torch.from_numpy(v), fg_bank, bg_bank) for k, v in siglip.items()}
meta["siglip"] = meta.key.map(dec)
meta["depth_bin"] = pd.cut(meta.depth_q25, [0, 1.0, 1.5, 2.5, 3.5, 5, 100])
pd.crosstab(meta.depth_bin, meta.siglip, normalize="index").round(2)

siglip,background,foreground,unknown
depth_bin,,,
"(0.0, 1.0]",0.24,0.31,0.45
"(1.0, 1.5]",0.94,0.02,0.04
"(1.5, 2.5]",0.99,0.00,0.01
"(2.5, 3.5]",1.00,0.00,0.00
"(3.5, 5.0]",1.00,0.00,0.00
"(5.0, 100.0]",1.00,0.00,0.00


In [24]:
dec

{'L_1782121838_p0000_chunk_005_000001_162_1105': 'background',
 'L_1782121838_p0000_chunk_005_000001_1021_1036': 'unknown',
 'L_1782121838_p0000_chunk_005_000001_304_998': 'background',
 'L_1782121838_p0000_chunk_005_000001_445_949': 'background',
 'L_1782121838_p0000_chunk_005_000001_881_860': 'background',
 'L_1782121838_p0000_chunk_005_000001_1496_1217': 'background',
 'L_1782121838_p0000_chunk_005_000001_1495_1229': 'background',
 'L_1782121838_p0000_chunk_005_000002_264_968': 'background',
 'L_1782121838_p0000_chunk_005_000002_1134_897': 'background',
 'L_1782121838_p0000_chunk_005_000002_387_979': 'background',
 'L_1782121838_p0000_chunk_005_000002_888_878': 'background',
 'L_1782121838_p0000_chunk_005_000002_849_877': 'background',
 'L_1782121838_p0000_chunk_005_000003_1240_953': 'background',
 'L_1782121838_p0000_chunk_005_000003_955_949': 'background',
 'L_1782121838_p0000_chunk_005_000003_412_1180': 'background',
 'L_1782121838_p0000_chunk_005_000003_955_948': 'background',
 

In [28]:
meta.key.iloc[0]

'L_1782121838_p0000_chunk_005_000001_162_1105'